<a href="https://colab.research.google.com/github/OlhaZahrebelna/Intelligent-Support-Ticket-Router-using-NLP/blob/main/notebook/04_Test_Error_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test Error Analysis: Linear SVM vs BERT

This notebook compares Linear SVM and BERT on the same fixed test set. It measures overall and class-level performance, identifies dominant confusion patterns, analyzes prediction margins, and selects representative cases for manual or LLM-assisted review.

## Analysis steps

1. Load and validate prediction artifacts.
2. Calculate overall test metrics.
3. Compare model agreement and outcome groups.
4. Analyze class-level performance.
5. Identify dominant confusion pairs.
6. Calculate and analyze SVM and BERT margins.
7. Inspect low-margin ambiguous cases.
8. Select potential label-review candidates.
9. Export analysis results.

## 1. Imports and configuration

In [1]:
from pathlib import Path
import re
import warnings

import joblib
import numpy as np
import pandas as pd
import spacy

from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, classification_report, f1_score

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42

DATA_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

BERT_PREDICTIONS_PATH = DATA_DIR / "bert_test_predictions.csv"
SVM_PREDICTIONS_PATH = DATA_DIR / "svm_test_predictions.csv"
SVM_MODEL_PATH = DATA_DIR / "svm_pipeline.joblib"

OUTPUT_DIR = DATA_DIR / "error_analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Compatibility class for loading the serialized SVM pipeline

The saved pipeline contains a custom `TextPreprocessor`. The class is defined here only so that `joblib` can deserialize the existing model artifact. In a production repository, this class should live in a reusable Python module.

In [2]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"],
)

NEGATION_WORDS = {
    "no",
    "not",
    "never",
    "cannot",
    "n't",
}


class TextPreprocessor(BaseEstimator, TransformerMixin):
    """
    Preprocessing transformer for English support-ticket texts.

    Steps:
    1. Convert values to strings.
    2. Normalize contractions and negations.
    3. Remove URLs and email addresses.
    4. Apply spaCy tokenization and lemmatization.
    5. Remove stop words, except meaningful negations.
    6. Keep alphabetic tokens only.
    """

    def __init__(
        self,
        min_token_len=2,
        batch_size=500,
    ):
        self.min_token_len = min_token_len
        self.batch_size = batch_size

    def fit(self, X, y=None):
        return self

    @staticmethod
    def normalize_text(text):
        """
        Normalize common English contractions before spaCy processing.
        """

        text = str(text)

        # Normalize apostrophes.
        text = text.replace("’", "'")
        text = text.replace("`", "'")

        # Important irregular negations.
        text = re.sub(
            r"\bcan't\b",
            "cannot",
            text,
            flags=re.IGNORECASE,
        )

        text = re.sub(
            r"\bwon't\b",
            "will not",
            text,
            flags=re.IGNORECASE,
        )

        text = re.sub(
            r"\bshan't\b",
            "shall not",
            text,
            flags=re.IGNORECASE,
        )

        # General negation contractions:
        # isn't -> is not
        # didn't -> did not
        # couldn't -> could not
        # doesn't -> does not
        text = re.sub(
            r"n['’]t\b",
            " not",
            text,
            flags=re.IGNORECASE,
        )

        # Remove URLs.
        text = re.sub(
            r"https?://\S+|www\.\S+",
            " ",
            text,
        )

        # Remove email addresses.
        text = re.sub(
            r"\b[\w.\-+]+@[\w.\-]+\.\w+\b",
            " ",
            text,
        )

        # Remove unnecessary whitespace.
        text = re.sub(r"\s+", " ", text).strip()

        return text

    def transform(self, X):
        """
        Transform an iterable of texts into cleaned strings.
        """

        texts = [
            self.normalize_text(text)
            if text is not None and not (
                isinstance(text, float) and np.isnan(text)
            )
            else ""
            for text in X
        ]

        processed_texts = []

        docs = nlp.pipe(
            texts,
            batch_size=self.batch_size,
        )

        for doc in docs:
            tokens = []

            for token in doc:
                token_text = token.text.lower().strip()
                lemma = token.lemma_.lower().strip()

                # spaCy can occasionally return an empty lemma.
                if not lemma:
                    continue

                # Preserve negation either by original token
                # or by its lemma.
                is_negation = (
                    token_text in NEGATION_WORDS
                    or lemma in NEGATION_WORDS
                )

                # Keep only alphabetic words and explicit negations.
                # "n't" is not alphabetic, so is_negation is checked separately.
                is_valid_token = token.is_alpha or is_negation

                if not is_valid_token:
                    continue

                # Remove stop words, except negations.
                if token.is_stop and not is_negation:
                    continue

                # Do not remove short negations such as "no".
                if len(lemma) < self.min_token_len and not is_negation:
                    continue

                # Normalize all negative forms to "not",
                # except "no" and "never", which preserve different meanings.
                if token_text in {"n't", "cannot"} or lemma == "cannot":
                    lemma = "not"

                tokens.append(lemma)

            processed_texts.append(" ".join(tokens))

        return np.asarray(processed_texts)

## 2. Load artifacts

In [3]:
required_paths = [
    BERT_PREDICTIONS_PATH,
    SVM_PREDICTIONS_PATH,
    SVM_MODEL_PATH,
]

missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required artifacts:\n- " + "\n- ".join(missing_paths)
    )

df_bert = pd.read_csv(BERT_PREDICTIONS_PATH)
df_svm = pd.read_csv(SVM_PREDICTIONS_PATH)
svm_model = joblib.load(SVM_MODEL_PATH)

print("BERT predictions:", df_bert.shape)
print("SVM predictions:", df_svm.shape)
print("SVM model loaded:", type(svm_model).__name__)

BERT predictions: (4750, 8)
SVM predictions: (4750, 5)
SVM model loaded: Pipeline


## 3. Validate and merge predictions

The two prediction files must represent exactly the same test records. All comparisons use `record_id`, not row order.

In [4]:
required_svm_columns = {
    "record_id",
    "text",
    "true_label",
    "svm_pred",
}

required_bert_columns = {
    "record_id",
    "text",
    "true_label",
    "bert_pred",
    "bert_confidence",
    "bert_margin",
}

missing_svm = required_svm_columns - set(df_svm.columns)
missing_bert = required_bert_columns - set(df_bert.columns)

if missing_svm:
    raise ValueError(f"Missing SVM columns: {sorted(missing_svm)}")
if missing_bert:
    raise ValueError(f"Missing BERT columns: {sorted(missing_bert)}")

if not df_svm["record_id"].is_unique:
    raise ValueError("SVM predictions contain duplicate record_id values.")
if not df_bert["record_id"].is_unique:
    raise ValueError("BERT predictions contain duplicate record_id values.")

if set(df_svm["record_id"]) != set(df_bert["record_id"]):
    raise ValueError("SVM and BERT files contain different test records.")

merge_df = df_svm.merge(
    df_bert,
    on="record_id",
    how="inner",
    suffixes=("_svm", "_bert"),
    validate="one_to_one",
)

if not merge_df["text_svm"].fillna("").eq(
    merge_df["text_bert"].fillna("")
).all():
    raise ValueError("Text values differ between prediction files.")

if not merge_df["true_label_svm"].eq(
    merge_df["true_label_bert"]
).all():
    raise ValueError("True labels differ between prediction files.")

merge_df = merge_df.drop(
    columns=["text_bert", "true_label_bert"]
).rename(
    columns={
        "text_svm": "text",
        "true_label_svm": "true_label",
    }
)

merge_df["svm_correct"] = merge_df["svm_pred"].eq(
    merge_df["true_label"]
)
merge_df["bert_correct"] = merge_df["bert_pred"].eq(
    merge_df["true_label"]
)
merge_df["models_agree"] = merge_df["svm_pred"].eq(
    merge_df["bert_pred"]
)

print("Merged test records:", len(merge_df))
print("Model agreement:", f"{merge_df['models_agree'].mean():.2%}")

Merged test records: 4750
Model agreement: 47.37%


## 4. Overall test performance

In [5]:
metrics_summary = pd.DataFrame(
    {
        "Model": ["Linear SVM", "BERT"],
        "Accuracy": [
            accuracy_score(
                merge_df["true_label"],
                merge_df["svm_pred"],
            ),
            accuracy_score(
                merge_df["true_label"],
                merge_df["bert_pred"],
            ),
        ],
        "Macro F1": [
            f1_score(
                merge_df["true_label"],
                merge_df["svm_pred"],
                average="macro",
            ),
            f1_score(
                merge_df["true_label"],
                merge_df["bert_pred"],
                average="macro",
            ),
        ],
        "Weighted F1": [
            f1_score(
                merge_df["true_label"],
                merge_df["svm_pred"],
                average="weighted",
            ),
            f1_score(
                merge_df["true_label"],
                merge_df["bert_pred"],
                average="weighted",
            ),
        ],
    }
)

display(metrics_summary.round(4))

,Model,Accuracy,Macro F1,Weighted F1
0,Linear SVM,0.5604,0.5427,0.5616
1,BERT,0.4288,0.4297,0.4293


The stronger primary router should be selected from these fixed-test metrics. The weaker model can still provide useful disagreement and audit signals, but it should not automatically receive equal production weight.

## 5. Model outcome groups

In [6]:
merge_df["outcome_group"] = np.select(
    [
        merge_df["svm_correct"] & merge_df["bert_correct"],
        merge_df["svm_correct"] & ~merge_df["bert_correct"],
        ~merge_df["svm_correct"] & merge_df["bert_correct"],
        ~merge_df["svm_correct"] & ~merge_df["bert_correct"],
    ],
    [
        "Both correct",
        "Only SVM correct",
        "Only BERT correct",
        "Both wrong",
    ],
    default="Unknown",
)

outcome_summary = (
    merge_df["outcome_group"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)

outcome_summary["Share"] = outcome_summary["Count"] / len(merge_df)
display(outcome_summary)

,Outcome,Count,Share
0,Both wrong,1606,0.338105
1,Both correct,1555,0.327368
2,Only SVM correct,1107,0.233053
3,Only BERT correct,482,0.101474


This table separates model-specific errors from shared errors. A large `Both wrong` group indicates that the main difficulty is not limited to one model architecture.

## 6. Class-level performance

In [7]:
svm_report = classification_report(
    merge_df["true_label"],
    merge_df["svm_pred"],
    output_dict=True,
    zero_division=0,
)

bert_report = classification_report(
    merge_df["true_label"],
    merge_df["bert_pred"],
    output_dict=True,
    zero_division=0,
)

labels = sorted(merge_df["true_label"].unique())

class_metrics = pd.DataFrame(
    {
        "Queue": labels,
        "Support": [svm_report[label]["support"] for label in labels],
        "SVM Precision": [svm_report[label]["precision"] for label in labels],
        "SVM Recall": [svm_report[label]["recall"] for label in labels],
        "SVM F1": [svm_report[label]["f1-score"] for label in labels],
        "BERT Precision": [bert_report[label]["precision"] for label in labels],
        "BERT Recall": [bert_report[label]["recall"] for label in labels],
        "BERT F1": [bert_report[label]["f1-score"] for label in labels],
    }
)

class_metrics["F1 Difference (SVM - BERT)"] = (
    class_metrics["SVM F1"] - class_metrics["BERT F1"]
)

display(
    class_metrics.sort_values("SVM F1").round(3).reset_index(drop=True)
)

,Queue,Support,SVM Precision,SVM Recall,SVM F1,BERT Precision,BERT Recall,BERT F1,F1 Difference (SVM - BERT)
0,Sales and Pre-Sales,145.0,0.335,0.614,0.433,0.333,0.517,0.405,0.028
1,General Inquiry,68.0,0.376,0.515,0.435,0.222,0.147,0.177,0.258
2,IT Support,567.0,0.487,0.459,0.472,0.289,0.459,0.354,0.118
3,Product Support,886.0,0.560,0.464,0.507,0.380,0.361,0.370,0.137
4,Customer Service,714.0,0.517,0.500,0.508,0.354,0.354,0.354,0.154
5,Returns and Exchanges,235.0,0.433,0.630,0.513,0.412,0.387,0.399,0.114
6,Human Resources,92.0,0.451,0.696,0.547,0.453,0.467,0.460,0.087
7,Technical Support,1372.0,0.645,0.558,0.598,0.487,0.356,0.411,0.187
8,Service Outages and Maintenance,187.0,0.521,0.727,0.607,0.558,0.620,0.587,0.020
9,Billing and Payments,484.0,0.792,0.820,0.806,0.770,0.787,0.778,0.028


## 7. Dominant confusion pairs

Counts are calculated on the full test set, not only on low-margin examples.

In [8]:
def confusion_table(
    frame,
    prediction_column,
    correct_column,
    top_n=15,
):
    return (
        frame.loc[
            ~frame[correct_column],
            ["true_label", prediction_column],
        ]
        .value_counts()
        .reset_index(name="Count")
        .rename(columns={prediction_column: "Predicted label"})
        .head(top_n)
    )


svm_confusions = confusion_table(
    merge_df,
    prediction_column="svm_pred",
    correct_column="svm_correct",
)

bert_confusions = confusion_table(
    merge_df,
    prediction_column="bert_pred",
    correct_column="bert_correct",
)

print("Top SVM confusion pairs")
display(svm_confusions)

print("Top BERT confusion pairs")
display(bert_confusions)

Top SVM confusion pairs


,true_label,Predicted label,Count
0,Product Support,Technical Support,162
1,Technical Support,Product Support,133
2,Technical Support,Customer Service,130
3,IT Support,Technical Support,122
4,Technical Support,IT Support,115
5,Product Support,Customer Service,77
6,Customer Service,Technical Support,74
7,Customer Service,Product Support,73
8,Product Support,IT Support,66
9,Technical Support,Returns and Exchanges,57


Top BERT confusion pairs


,true_label,Predicted label,Count
0,Technical Support,IT Support,367
1,Product Support,Technical Support,216
2,Technical Support,Product Support,206
3,Technical Support,Customer Service,158
4,Customer Service,Product Support,149
5,IT Support,Technical Support,130
6,Product Support,Customer Service,122
7,Product Support,IT Support,110
8,Customer Service,Technical Support,99
9,Customer Service,IT Support,80


In [9]:
shared_wrong = merge_df[
    ~merge_df["svm_correct"]
    & ~merge_df["bert_correct"]
].copy()

same_wrong_prediction = shared_wrong[
    shared_wrong["svm_pred"].eq(shared_wrong["bert_pred"])
].copy()

shared_confusions = (
    same_wrong_prediction[
        ["true_label", "svm_pred"]
    ]
    .value_counts()
    .reset_index(name="Count")
    .rename(columns={"svm_pred": "Shared prediction"})
)

print("Both models wrong:", len(shared_wrong))
print(
    "Both models select the same wrong label:",
    len(same_wrong_prediction),
)
display(shared_confusions.head(15))

Both models wrong: 1606
Both models select the same wrong label: 695


,true_label,Shared prediction,Count
0,Product Support,Technical Support,86
1,Technical Support,IT Support,68
2,Technical Support,Product Support,55
3,IT Support,Technical Support,47
4,Technical Support,Customer Service,39
5,Customer Service,Product Support,36
6,Customer Service,Technical Support,32
7,Product Support,IT Support,27
8,Product Support,Customer Service,27
9,Technical Support,Service Outages and Maintenance,22


### Taxonomy of the most frequently confused queues

Use this table only to interpret confusion pairs that actually appear in the results above.

| Queue | What belongs here | What does not belong here | Typical markers |
|---|---|---|---|
| IT Support | Access, account, device, network, and credential issues | Product functionality issues | login, password, VPN |
| Product Support | Functionality of a specific product or feature | Infrastructure or access issues | feature, integration, configuration |
| Technical Support | General technical problems and malfunctions | Billing or sales requests | error, crash, malfunction |
| Customer Service | Complaints, service experience, and status requests | Detailed technical diagnosis | complaint, delayed response |
| Service Outages and Maintenance | Widespread service unavailability or planned maintenance | Problems affecting only one user | outage, downtime, unavailable for everyone |

Frequent errors among these queues may indicate overlapping definitions, insufficient ticket context, or inconsistent annotation. The confusion counts alone do not prove that labels are incorrect.

## 8. Validate the loaded SVM pipeline

In [10]:
loaded_predictions = svm_model.predict(merge_df["text"])

match_rate = np.mean(
    loaded_predictions == merge_df["svm_pred"].to_numpy()
)

print("Saved model / CSV prediction match:", f"{match_rate:.2%}")

if match_rate < 1.0:
    warnings.warn(
        "The loaded model does not fully reproduce the SVM prediction file. "
        "Check model version, preprocessing, and test split provenance."
    )

Saved model / CSV prediction match: 100.00%


## 9. Calculate SVM margins

For multiclass Linear SVM, the margin is the difference between the highest and second-highest decision scores. Larger margins indicate stronger separation between the top two predicted classes, but they are not calibrated probabilities.

In [11]:
decision_scores = svm_model.decision_function(merge_df["text"])

if decision_scores.ndim != 2:
    raise ValueError(
        "Expected multiclass decision scores with shape "
        "(n_samples, n_classes)."
    )

classifier = svm_model.named_steps.get("classifier")
if classifier is None or not hasattr(classifier, "classes_"):
    raise AttributeError(
        "Could not find classifier.classes_ in the saved pipeline."
    )

svm_classes = classifier.classes_
sorted_indices = np.argsort(decision_scores, axis=1)

top_indices = sorted_indices[:, -1]
second_indices = sorted_indices[:, -2]
rows = np.arange(len(merge_df))

merge_df["svm_top_score"] = decision_scores[rows, top_indices]
merge_df["svm_second_score"] = decision_scores[rows, second_indices]
merge_df["svm_margin"] = (
    merge_df["svm_top_score"] - merge_df["svm_second_score"]
)
merge_df["svm_second_pred"] = svm_classes[second_indices]

reconstructed_top_pred = svm_classes[top_indices]
if not np.array_equal(
    reconstructed_top_pred,
    merge_df["svm_pred"].to_numpy(),
):
    raise ValueError(
        "Top SVM decision scores do not reproduce svm_pred."
    )

display(
    merge_df[
        [
            "record_id",
            "true_label",
            "svm_pred",
            "svm_second_pred",
            "svm_margin",
        ]
    ].head()
)

,record_id,true_label,svm_pred,svm_second_pred,svm_margin
0,19592,IT Support,IT Support,Customer Service,0.286159
1,15528,Customer Service,IT Support,Technical Support,0.042213
2,23498,Billing and Payments,Billing and Payments,IT Support,1.684428
3,19877,Technical Support,Technical Support,Returns and Exchanges,0.283492
4,6360,Technical Support,Returns and Exchanges,Customer Service,0.164168


## 10. Low-margin ambiguity analysis

The bottom 10% of margins within each model is used as a relative exploratory threshold. It is not a calibrated confidence threshold, and low margins alone do not indicate incorrect labels.

In [12]:
LOW_MARGIN_QUANTILE = 0.10

svm_low_threshold = merge_df["svm_margin"].quantile(
    LOW_MARGIN_QUANTILE
)
bert_low_threshold = merge_df["bert_margin"].quantile(
    LOW_MARGIN_QUANTILE
)

merge_df["svm_low_margin"] = merge_df["svm_margin"].le(
    svm_low_threshold
)
merge_df["bert_low_margin"] = merge_df["bert_margin"].le(
    bert_low_threshold
)
merge_df["both_models_low_margin"] = (
    merge_df["svm_low_margin"]
    & merge_df["bert_low_margin"]
)

print("SVM low-margin threshold:", round(svm_low_threshold, 4))
print("BERT low-margin threshold:", round(bert_low_threshold, 4))
print(
    "Records low-margin for both models:",
    int(merge_df["both_models_low_margin"].sum()),
)

SVM low-margin threshold: 0.0309
BERT low-margin threshold: 0.029
Records low-margin for both models: 62


In [13]:
both_low_margin = merge_df[
    merge_df["both_models_low_margin"]
].copy()

low_margin_summary = pd.DataFrame(
    {
        "Case type": [
            "Models agree",
            "Models disagree",
            "Both correct",
            "Only SVM correct",
            "Only BERT correct",
            "Both wrong",
        ],
        "Count": [
            both_low_margin["models_agree"].sum(),
            (~both_low_margin["models_agree"]).sum(),
            (
                both_low_margin["svm_correct"]
                & both_low_margin["bert_correct"]
            ).sum(),
            (
                both_low_margin["svm_correct"]
                & ~both_low_margin["bert_correct"]
            ).sum(),
            (
                ~both_low_margin["svm_correct"]
                & both_low_margin["bert_correct"]
            ).sum(),
            (
                ~both_low_margin["svm_correct"]
                & ~both_low_margin["bert_correct"]
            ).sum(),
        ],
    }
)

if len(both_low_margin):
    low_margin_summary["Share"] = (
        low_margin_summary["Count"] / len(both_low_margin)
    )
else:
    low_margin_summary["Share"] = np.nan

display(low_margin_summary)

,Case type,Count,Share
0,Models agree,14,0.225806
1,Models disagree,48,0.774194
2,Both correct,4,0.064516
3,Only SVM correct,7,0.112903
4,Only BERT correct,15,0.241935
5,Both wrong,36,0.580645


In [14]:
display(
    both_low_margin[
        [
            "record_id",
            "text",
            "true_label",
            "svm_pred",
            "svm_second_pred",
            "svm_margin",
            "bert_pred",
            "bert_confidence",
            "bert_margin",
            "outcome_group",
        ]
    ]
    .sort_values(["svm_margin", "bert_margin"])
    .head(20)
)

,record_id,text,true_label,svm_pred,svm_second_pred,svm_margin,bert_pred,bert_confidence,bert_margin,outcome_group
2040,20734,"Respected Customer Support Team, I am contacting you to bring to your notice a data breach that has taken place. This may be due to the system's Elasticsearch being out of dat...",Returns and Exchanges,Billing and Payments,Product Support,0.000434,Technical Support,0.325363,0.006916,Both wrong
400,6931,"Boost Digital Brand Expansion request for enhancements in digital strategies aimed at brand growth through targeted marketing, optimized product support, and initiatives to imp...",Product Support,Sales and Pre-Sales,Product Support,0.000595,Customer Service,0.303239,0.002360,Both wrong
3834,15878,Traffic Support Issue Noticed a decline in web traffic and engagement metrics. This could be due to an algorithm change or a technical glitch. Cache was cleared and content was...,Technical Support,Technical Support,General Inquiry,0.000918,Technical Support,0.319591,0.006793,Both correct
2849,12760,Boosting Digital Brand Identity Is it possible to get more information about your digital strategy services?,Product Support,Returns and Exchanges,Billing and Payments,0.000925,Billing and Payments,0.307393,0.024486,Both wrong
4432,13707,Marketing Challenge A marketing firm observed a decline in brand engagement. This might be due to algorithm changes or ineffective ad targeting. After reviewing ad performance ...,Customer Service,Product Support,Customer Service,0.001055,Product Support,0.353109,0.012198,Both wrong
2656,13661,"Security Concern at Hospital System Customer Support, facing a security issue with the hospital's system. There have been unauthorized access attempts to medical data, posing a...",Product Support,Customer Service,Product Support,0.001120,Returns and Exchanges,0.220436,0.026471,Both wrong
4646,19186,"Request for Assistance with Data Analytics Tools Malfunction Dear Customer Support, I am contacting you to report an issue with our data analytics tools. The tools have malfunc...",IT Support,IT Support,Returns and Exchanges,0.001958,Technical Support,0.329977,0.010776,Only SVM correct
2243,17223,Support for Integrating Drupal with Magento I require assistance in the process of integrating Drupal with Magento to enhance our e-commerce platform. Could you offer detailed ...,Product Support,Returns and Exchanges,Product Support,0.003480,Billing and Payments,0.179117,0.003121,Both wrong
3427,22353,"Medical Data Breaches Resulting from Insufficient Security Measures Hello Customer Support, I am contacting you to address a critical problem involving medical data breaches ca...",Technical Support,Returns and Exchanges,Technical Support,0.003675,Customer Service,0.221362,0.007856,Both wrong
1068,20901,"Improvement of Digital Brand Visibility Hello from Customer Support, we are here to discuss digital strategies that can enhance your brand's visibility for your products. We wo...",Product Support,General Inquiry,Customer Service,0.003948,Customer Service,0.361274,0.011115,Both wrong


## 11. Potential label-review candidates

The strongest automatic review signal used here is:

- both models predict the same alternative label;
- that label differs from the original label;
- both models are in the upper quartile of their respective margins.

These records are review candidates, not confirmed annotation errors.

In [15]:
HIGH_MARGIN_QUANTILE = 0.75

svm_high_threshold = merge_df["svm_margin"].quantile(
    HIGH_MARGIN_QUANTILE
)
bert_high_threshold = merge_df["bert_margin"].quantile(
    HIGH_MARGIN_QUANTILE
)

merge_df["models_agree_on_alternative"] = (
    merge_df["models_agree"]
    & ~merge_df["svm_correct"]
)

merge_df["potential_label_issue"] = (
    merge_df["models_agree_on_alternative"]
    & merge_df["svm_margin"].ge(svm_high_threshold)
    & merge_df["bert_margin"].ge(bert_high_threshold)
)

label_review_candidates = (
    merge_df[merge_df["potential_label_issue"]]
    .copy()
    .sort_values(
        ["svm_margin", "bert_margin"],
        ascending=False,
    )
)

print("SVM high-margin threshold:", round(svm_high_threshold, 4))
print("BERT high-margin threshold:", round(bert_high_threshold, 4))
print("Label-review candidates:", len(label_review_candidates))

SVM high-margin threshold: 0.3978
BERT high-margin threshold: 0.3543
Label-review candidates: 35


In [16]:
display(
    label_review_candidates[
        [
            "record_id",
            "text",
            "true_label",
            "svm_pred",
            "bert_pred",
            "svm_margin",
            "bert_confidence",
            "bert_margin",
        ]
    ].head(20)
)

,record_id,text,true_label,svm_pred,bert_pred,svm_margin,bert_confidence,bert_margin
583,9139,Experienced Service Outage Impacting Project Management Tools Integration We are facing service outages that are affecting the integration of project management tools such as A...,Product Support,Service Outages and Maintenance,Service Outages and Maintenance,1.518942,0.990229,0.987972
2157,2712,Regular Service Interruptions During Busy Hours Regular service interruptions occur during peak times.,Technical Support,Service Outages and Maintenance,Service Outages and Maintenance,1.238379,0.982507,0.978241
41,14177,"Issue with Service Disruptions Noted Dear Customer Support, I am encountering service disruptions across various products, such as IFTTT and DataRobot, which might be due to un...",Technical Support,Service Outages and Maintenance,Service Outages and Maintenance,1.088087,0.781118,0.659205
1333,5520,Digital Campaigns Encountering Downtime Problems Facing unforeseen outages; tried to reboot the systems.,Technical Support,Service Outages and Maintenance,Service Outages and Maintenance,0.987856,0.679210,0.549580
696,1267,"Enhancing Investment Strategies with Data Analytics Customer Support, I am reaching out to inquire about the data analytics services your firm offers. Could you please provide ...",Returns and Exchanges,Customer Service,Customer Service,0.953689,0.528184,0.386925
4610,16302,"Service Disruptions Encountered service interruptions across various products, specifically the Smart-Thermostat Polk Audio Signa S2, following the latest software update.",Technical Support,Service Outages and Maintenance,Service Outages and Maintenance,0.913172,0.981566,0.977547
2827,23258,Dynamics 365 Support Which tools can enhance investment returns?,Billing and Payments,Returns and Exchanges,Returns and Exchanges,0.857113,0.721420,0.527631
266,10915,"Encountered service disruption impacting integration with project management tools such as ActiveCampaign, Xero, and MongoDB 4.4. Recent updates might have caused conflicting ...",Product Support,Service Outages and Maintenance,Service Outages and Maintenance,0.746205,0.969794,0.962214
128,3279,"Digital Campaigns Encountering Outages Facing unforeseen downtime, efforts made to restart the system",Technical Support,Service Outages and Maintenance,Service Outages and Maintenance,0.701598,0.935070,0.910248
131,18459,Support for Marketing Campaign We are inquiring about the recent marketing campaign that did not meet expectations due to low engagement. We have attempted to enhance the campa...,Customer Service,Returns and Exchanges,Returns and Exchanges,0.683530,0.864816,0.815355


In [17]:
label_review_pairs = (
    label_review_candidates[
        ["true_label", "svm_pred"]
    ]
    .value_counts()
    .reset_index(name="Count")
    .rename(columns={"svm_pred": "Shared alternative label"})
)

display(label_review_pairs.head(15))

,true_label,Shared alternative label,Count
0,Technical Support,Service Outages and Maintenance,8
1,Technical Support,IT Support,7
2,Product Support,Service Outages and Maintenance,3
3,Product Support,IT Support,3
4,IT Support,Service Outages and Maintenance,2
5,Technical Support,Billing and Payments,2
6,IT Support,Customer Service,1
7,Billing and Payments,Returns and Exchanges,1
8,Product Support,Sales and Pre-Sales,1
9,Customer Service,Product Support,1


## 12. Text-length analysis

This checks whether shorter tickets are systematically associated with model errors. Text length is descriptive only and should not be treated as a causal explanation.

In [18]:
merge_df["text_word_count"] = (
    merge_df["text"]
    .fillna("")
    .str.split()
    .str.len()
)

text_length_summary = (
    merge_df.groupby("outcome_group")["text_word_count"]
    .agg(["count", "median", "mean"])
    .reset_index()
    .rename(
        columns={
            "outcome_group": "Outcome",
            "count": "Count",
            "median": "Median words",
            "mean": "Mean words",
        }
    )
)

display(text_length_summary.round(2))

,Outcome,Count,Median words,Mean words
0,Both correct,1555,61.0,61.53
1,Both wrong,1606,58.0,59.40
2,Only BERT correct,482,53.0,57.20
3,Only SVM correct,1107,64.0,62.79


## 13. Representative error cases

Representative records are sampled from the major outcome groups. These examples support qualitative inspection without dumping the full test set into the notebook.

In [19]:
REPRESENTATIVE_COLUMNS = [
    "record_id",
    "text",
    "true_label",
    "svm_pred",
    "bert_pred",
    "svm_margin",
    "bert_margin",
]

representative_cases = (
    merge_df[
        merge_df["outcome_group"].isin(
            ["Only SVM correct", "Only BERT correct", "Both wrong"]
        )
    ]
    .groupby("outcome_group", group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(5, len(group)),
            random_state=RANDOM_STATE,
        )
    )
    .reset_index(drop=True)
)

display(
    representative_cases[
        ["outcome_group"] + REPRESENTATIVE_COLUMNS
    ]
)

/tmp/ipykernel_7322/2752929820.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,outcome_group,record_id,text,true_label,svm_pred,bert_pred,svm_margin,bert_margin
0,Both wrong,11580,"A data breach has taken place, potentially endangering sensitive medical records within the hospital's systems. This might be due to inadequate cybersecurity measures. Despite...",IT Support,Technical Support,Human Resources,0.071498,0.094920
1,Both wrong,12141,Request for Integration of Project Management Features Could you please integrate project management features into the affected products to enhance collaboration? We believe th...,IT Support,Customer Service,Product Support,0.005678,0.088880
2,Both wrong,13040,"Support Required for Integrating CCleaner Hello customer support team, we are seeking assistance with integrating the CCleaner project management SaaS platform. Could you offer...",Technical Support,IT Support,Customer Service,0.132607,0.060390
3,Both wrong,14182,"Support for Docker Looking to integrate Docker 20.10 into our project management SaaS solution. Could you provide details on the integration process, necessary configurations, ...",Technical Support,Customer Service,IT Support,0.182744,0.196057
4,Both wrong,21022,"Strategies for Digital Brand Growth with RapidMiner and BigCommerce Hello Customer Support, I am contactin' to seek informati'on on the digital tactics that support brand growt...",Product Support,Sales and Pre-Sales,IT Support,0.190460,0.377798
5,Only BERT correct,2669,"Critical Outage in SaaS Platform The project management component of the SaaS platform has encountered a significant outage, impacting several interconnected products. The like...",Product Support,Service Outages and Maintenance,Product Support,0.440146,0.148589
6,Only BERT correct,14487,Request for Information on Integrating Smartsheet Scalable SaaS Project Management Could you provide detailed information on integrating Smartsheet's scalable SaaS project mana...,Customer Service,Sales and Pre-Sales,Customer Service,0.097902,0.322205
7,Only BERT correct,21659,Problem with Video Rendering in Final Cut Pro X Video rendering in Final Cut Pro X is failing due to insufficient memory and software conflicts. I have restarted the software a...,IT Support,Technical Support,IT Support,0.075196,0.392468
8,Only BERT correct,2056,"Support Request for IBM SPSS I am reaching out to inquire about the comprehensive product features and integration options available for IBM SPSS Statistics 28, specifically in...",Sales and Pre-Sales,Customer Service,Sales and Pre-Sales,0.357513,0.188045
9,Only BERT correct,18271,"Support for Boosting Brand Visibility Hi, could you share more about digital strategies that can improve brand visibility?",Billing and Payments,Returns and Exchanges,Billing and Payments,0.165846,0.274401


## 14. Key findings and routing implications

### Model performance

- **Linear SVM is the stronger primary classifier.** It achieved  a Macro F1 of **0.5427**, compared with **0.4297** Macro F1 for BERT.
- SVM was the only correct model for **1,107 records (23.31%)**, while BERT was uniquely correct for **482 records (10.15%)**. SVM therefore solved more than twice as many cases that BERT missed.
- Both models were correct for **1,555 records (32.74%)**, but both failed on **1,606 records (33.81%)**. The large shared-error group shows that many errors are not specific to one model architecture.

### Class-level weaknesses

- **Billing and Payments** was the strongest class for both models: SVM F1 was **0.806**, and BERT F1 was **0.778**.
- **Service Outages and Maintenance** was also comparatively well separated, with SVM F1 of **0.607** and BERT F1 of **0.587**.
- The largest SVM advantage appeared for **General Inquiry**, where SVM F1 was **0.435** and BERT F1 was only **0.177**.
- BERT also underperformed substantially on **Technical Support**, **Customer Service**, **Product Support**, and **IT Support**.
- The weakest SVM results were concentrated in **Sales and Pre-Sales**, **General Inquiry**, and **IT Support**, whose F1 scores remained below **0.48**.

### Dominant error patterns

- The main errors are concentrated among semantically related queues: **Technical Support**, **Product Support**, **IT Support**, and **Customer Service**.
- The most frequent SVM confusion was **Product Support → Technical Support** with **162 cases**.
- BERT showed an especially strong bias toward **IT Support** for true **Technical Support** tickets, producing **367 such errors**.
- Both models selected the same wrong label for **695 of the 1,606 shared errors**. The most common shared confusion was **Product Support → Technical Support** with **86 cases**.
- These patterns indicate that the boundary between the technical queues is a central source of error and should be clarified in the label taxonomy.

### Margin and review analysis

- Only **62 records** were in the bottom 10% of margins for both models.
- Among these jointly low-margin cases, **36 records (58.06%)** were misclassified by both models, while only **4 records (6.45%)** were correctly classified by both.
- Jointly low margins are therefore useful for identifying ambiguous cases, but they cover only a small part of the total error set and should not be treated as evidence of incorrect annotation.
- The high-margin agreement rule identified **35 records** for label or taxonomy review. These are review candidates, not confirmed labeling errors.
- The most frequent high-confidence review patterns were **Technical Support → Service Outages and Maintenance** and **Technical Support → IT Support**.

### Text length

- Text length differed only moderately across outcome groups.
- Tickets misclassified by both models had a median length of **58 words**, compared with **61 words** for tickets correctly classified by both.
- Ticket length alone therefore does not explain the observed performance gap. Queue overlap is a more important analytical direction.

### Recommendation for the routing system

- Use **Linear SVM as the primary router**.
- Do not use the current BERT model as an equal-vote ensemble component because it is materially weaker on the fixed test set.
- Route only selected cases to an additional agent:
  - low-margin SVM predictions;
  - top-two predictions belonging to known confusion pairs;
  - tickets involving overlapping technical queues;
  - high-confidence model agreement that contradicts the original label during offline audit.
- The agent should choose between a restricted set of candidate queues using explicit taxonomy definitions. It should not independently classify every ticket across all queues.
